In [1]:
import os
import torch
import pandas as pd
import numpy as np
from transformers import AutoImageProcessor, AutoModelForImageClassification
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report
from torchvision.transforms import (
    CenterCrop,
    Compose,
    Normalize,
    Resize,
    ToTensor,
)
import gc
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

In [2]:
DATASET_PATHS = {
    "ham": "/data/team01/ds340w/datasets/ham10000_split",
    "atlas_isic": "/data/team01/ds340w/datasets/Atlas_ISIC_split",
}

# Model paths - includes both original and GLIDE-augmented models
MODEL_SAVE_PATHS = {
    "dino": {
        "ham": "/data/team01/ds340w/Aiden-Enhanced/models/model_saves/dino.ham-finetuned-SkinDisease",
        "ham_glide": "/data/team01/ds340w/Aiden-Enhanced/models/model_saves/dino.ham.glide-finetuned-SkinDisease",
        "atlas_isic": "/data/team01/ds340w/Aiden-Enhanced/models/model_saves/dino.atlas_isic-finetuned-SkinDisease",
        "atlas_isic_glide": "/data/team01/ds340w/Aiden-Enhanced/models/model_saves/dino.atlas_isic.glide-finetuned-SkinDisease"
    },
    "swin": {
        "ham": "/data/team01/ds340w/Aiden-Enhanced/models/model_saves/swin.ham-finetuned-SkinDisease",
        "ham_glide": "/data/team01/ds340w/Aiden-Enhanced/models/model_saves/swin.ham.glide-finetuned-SkinDisease",
        "atlas_isic": "/data/team01/ds340w/Aiden-Enhanced/models/model_saves/swin.atlas_isic-base-patch4-window7-224-finetuned-SkinDisease",
        "atlas_isic_glide": "/data/team01/ds340w/Aiden-Enhanced/models/model_saves/swin.atlas_isic.glide-finetuned-SkinDisease",
    },
    "vit": {
        "ham": "/data/team01/ds340w/Aiden-Enhanced/models/model_saves/vit.ham-finetuned-SkinDisease",
        "ham_glide": "/data/team01/ds340w/Aiden-Enhanced/models/model_saves/vit.ham.glide-finetuned-SkinDisease",
        "atlas_isic": "/data/team01/ds340w/Aiden-Enhanced/models/model_saves/vit.atlas_isic-base-patch16-224-finetuned-SkinDisease",
        "atlas_isic_glide": "/data/team01/ds340w/Aiden-Enhanced/models/model_saves/vit.atlas_isic.glide-finetuned-SkinDisease",
    }
}

# Class names for each dataset
CLASS_NAMES = {
    "ham": ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc'],
    "atlas_isic": ['Actinic Keratosis', 'Basal Cell Carcinoma', 'Darier_s Disease', 'Dermatofibroma', 
                   'Epidermolysis Bullosa Pruriginosa', 'Hailey-Hailey Disease', 'Herpes Simplex', 
                   'Impetigo', 'Larva Migrans', 'Leprosy Borderline', 'Leprosy Lepromatous', 
                   'Leprosy Tuberculoid', 'Lichen Planus', 'Lupus Erythematosus Chronicus Discoides', 
                   'Melanoma', 'Molluscum Contagiosum', 'Mycosis Fungoides', 'Neurofibromatosis', 
                   'Nevus', 'Papilomatosis Confluentes And Reticulate', 'Pediculosis Capitis', 
                   'Pigmented Benign Keratosis', 'Pityriasis Rosea', 'Porokeratosis Actinic', 
                   'Psoriasis', 'Seborrheic keratosis', 'Squamous cell carcinoma', 'Tinea Corporis', 
                   'Tinea Nigra', 'Tungiasis', 'Vascular lesion'],
}

In [3]:
def clear_gpu():
    """Clear GPU cache"""
    torch.cuda.empty_cache()
    gc.collect()

def get_transforms(image_processor):
    """Get validation transforms"""
    normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std)
    
    if "height" in image_processor.size:
        size = (image_processor.size["height"], image_processor.size["width"])
        crop_size = size
    elif "shortest_edge" in image_processor.size:
        size = image_processor.size["shortest_edge"]
        crop_size = (size, size)
    
    val_transforms = Compose([
        Resize(size),
        CenterCrop(crop_size),
        ToTensor(),
        normalize,
    ])
    
    return val_transforms


In [4]:
def evaluate_model(model_name, dataset_name, model_path, dataset_path, gpu_id):
    """Evaluate a single model on a dataset"""
    try:
        print(f"\n{'='*60}")
        print(f"Evaluating: {model_name} on {dataset_name}")
        if ".glide" in model_path:
            print(f"Model type: GLIDE-augmented training")
        else:
            print(f"Model type: Original training")
        print(f"{'='*60}")
        
        # Set GPU
        os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_id}"
        print(f"Running on CUDA GPU: {gpu_id}")
        clear_gpu()
        
        # Load model and processor
        print("Loading model...")
        image_processor = AutoImageProcessor.from_pretrained(model_path)
        model = AutoModelForImageClassification.from_pretrained(model_path)
        
        # Load test dataset (using original test set)
        print(f"Loading test dataset from: {dataset_path}/test")
        test_dataset = load_dataset("imagefolder", data_dir=f"{dataset_path}/test")
        
        # Setup transforms
        val_transforms = get_transforms(image_processor)
        
        def preprocess_val(example_batch):
            example_batch["pixel_values"] = [
                val_transforms(image.convert("RGB")) for image in example_batch["image"]
            ]
            return example_batch
        
        test_dataset.set_transform(preprocess_val)  # Use the same validation transforms
        # Create a test dataset
        test_ds = test_dataset["train"]  # Use the "train" split for the test data
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        model.eval() # Good practice: set model to evaluation mode
        print(f"Using device: {device}")
        
        # Initialize lists to store predicted and actual labels
        predicted_labels = []
        actual_labels = []
        
        # Iterate through the test dataset and make predictions
        for example in test_ds:
            image = example["image"]
            
            # 2. Process the image (This creates tensors on the CPU by default)
            encoding = image_processor(image.convert("RGB"), return_tensors="pt")
        
            # 3. MOVE THE IMAGES TO GPU
            # We loop through the dictionary and move each tensor to the GPU
            encoding = {k: v.to(device) for k, v in encoding.items()}
        
            with torch.no_grad():
                # 4. Now both the model and the encoding are on the same device
                outputs = model(**encoding)
                logits = outputs.logits
        
            # 5. Bring the result back to CPU to extract the number
            predicted_class_idx = logits.argmax(-1).item()
            predicted_labels.append(predicted_class_idx)
            actual_labels.append(example["label"])
        
        # Calculate accuracy
        accuracy = accuracy_score(actual_labels, predicted_labels)
        class_names = CLASS_NAMES.get(dataset_name, [f"class_{i}" for i in range(len(set(actual_labels)))])
        report = classification_report(actual_labels, predicted_labels, 
                                      target_names=class_names, 
                                      output_dict=True)
        
        # Clear GPU memory
        del model
        clear_gpu()
        
        # Determine model variant for display
        variant = "GLIDE" if "glide" in model_path else "Original"
        
        return {
            "model": model_name,
            "dataset": dataset_name,
            "variant": variant,
            "model_key": f"{model_name}_{variant}",
            "accuracy": accuracy,
            "macro_precision": report["macro avg"]["precision"],
            "macro_recall": report["macro avg"]["recall"],
            "macro_f1": report["macro avg"]["f1-score"],
            "weighted_f1": report["weighted avg"]["f1-score"],
            "num_samples": len(test_ds)
        }
        
    except Exception as e:
        print(f"Error evaluating {model_name} on {dataset_name}: {str(e)}")
        return None

In [5]:
def main():
    """Main evaluation function"""
    # Define all combinations to evaluate
    combinations = []
    
    for model_name in ["dino", "swin", "vit"]:
        for dataset_name in ["ham", "atlas_isic"]:  # Only original test datasets
            # Check both original and GLIDE model variants
            for variant in ["", "_glide"]:
                model_key = f"{dataset_name}{variant}"
                if model_key in MODEL_SAVE_PATHS[model_name]:
                    model_path = MODEL_SAVE_PATHS[model_name][model_key]
                    dataset_path = DATASET_PATHS[dataset_name]
                    
                    if model_path and dataset_path and os.path.exists(model_path):
                        combinations.append((model_name, dataset_name, model_path, dataset_path))
                    else:
                        print(f"Skipping {model_name} on {dataset_name}{variant} - path not found")
    
    print(f"\nFound {len(combinations)} model-dataset combinations to evaluate")
    print(f"Each model will be evaluated on original test sets only\n")
    
    # Evaluate each combination
    results = []
    gpu_id = 3  # Use GPU 0, adjust as needed

    print(combinations)
    
    for model_name, dataset_name, model_path, dataset_path in combinations:
        #                       model_name, dataset_name, model_path, dataset_path, gpu_id
        result = evaluate_model(model_name, dataset_name, model_path, dataset_path, gpu_id)
        if result:
            results.append(result)
    
    # Create comparison tables
    if results:
        df = pd.DataFrame(results)
        
        # Pivot table for Accuracy - comparing Original vs GLIDE
        print("\n" + "="*80)
        print("ACCURACY COMPARISON: Original Training vs GLIDE-Augmented Training")
        print("="*80)
        pivot_acc = df.pivot_table(index="model", columns=["dataset", "variant"], values="accuracy")
        print(pivot_acc.round(4))
        
        print("\n" + "="*80)
        print("MACRO F1 SCORE COMPARISON: Original vs GLIDE")
        print("="*80)
        pivot_f1 = df.pivot_table(index="model", columns=["dataset", "variant"], values="macro_f1")
        print(pivot_f1.round(4))
        
        print("\n" + "="*80)
        print("WEIGHTED F1 SCORE COMPARISON: Original vs GLIDE")
        print("="*80)
        pivot_wf1 = df.pivot_table(index="model", columns=["dataset", "variant"], values="weighted_f1")
        print(pivot_wf1.round(4))
        
        # Create side-by-side comparison for each dataset
        for dataset in ["ham", "atlas_isic"]:
            print(f"\n{'='*80}")
            print(f"RESULTS FOR {dataset.upper()} DATASET")
            print(f"{'='*80}")
            dataset_df = df[df["dataset"] == dataset]
            comparison = dataset_df.pivot(index="model", columns="variant", values="accuracy")
            comparison["Improvement"] = comparison.get("GLIDE", 0) - comparison.get("Original", 0)
            print(comparison.round(4))
        
        # Save full results
        df.to_csv("model_evaluation_full_results.csv", index=False)
        
        # Save separate comparison tables
        pivot_acc.to_csv("model_comparison_accuracy.csv")
        pivot_f1.to_csv("model_comparison_macro_f1.csv")
        pivot_wf1.to_csv("model_comparison_weighted_f1.csv")
        
        print("\n" + "="*80)
        print("Results saved to CSV files:")
        print("  - model_evaluation_full_results.csv")
        print("  - model_comparison_accuracy.csv")
        print("  - model_comparison_macro_f1.csv")
        print("  - model_comparison_weighted_f1.csv")
        print("="*80)
        
        # Print detailed results
        print("\n" + "="*80)
        print("DETAILED RESULTS")
        print("="*80)
        for result in results:
            print(f"\n{result['model'].upper()} ({result['variant']}) on {result['dataset']}:")
            print(f"  Accuracy: {result['accuracy']:.4f}")
            print(f"  Macro F1: {result['macro_f1']:.4f}")
            print(f"  Weighted F1: {result['weighted_f1']:.4f}")
            print(f"  Samples: {result['num_samples']}")
    
    else:
        print("No successful evaluations completed!")


main()


Found 12 model-dataset combinations to evaluate
Each model will be evaluated on original test sets only

[('dino', 'ham', '/data/team01/ds340w/Aiden-Enhanced/models/model_saves/dino.ham-finetuned-SkinDisease', '/data/team01/ds340w/datasets/ham10000_split'), ('dino', 'ham', '/data/team01/ds340w/Aiden-Enhanced/models/model_saves/dino.ham.glide-finetuned-SkinDisease', '/data/team01/ds340w/datasets/ham10000_split'), ('dino', 'atlas_isic', '/data/team01/ds340w/Aiden-Enhanced/models/model_saves/dino.atlas_isic-finetuned-SkinDisease', '/data/team01/ds340w/datasets/Atlas_ISIC_split'), ('dino', 'atlas_isic', '/data/team01/ds340w/Aiden-Enhanced/models/model_saves/dino.atlas_isic.glide-finetuned-SkinDisease', '/data/team01/ds340w/datasets/Atlas_ISIC_split'), ('swin', 'ham', '/data/team01/ds340w/Aiden-Enhanced/models/model_saves/swin.ham-finetuned-SkinDisease', '/data/team01/ds340w/datasets/ham10000_split'), ('swin', 'ham', '/data/team01/ds340w/Aiden-Enhanced/models/model_saves/swin.ham.glide-fin

Loading weights:   0%|          | 0/225 [00:00<?, ?it/s]

Loading test dataset from: /data/team01/ds340w/datasets/ham10000_split/test


Resolving data files:   0%|          | 0/1001 [00:00<?, ?it/s]

Using device: cuda

Evaluating: dino on ham
Model type: GLIDE-augmented training
Running on CUDA GPU: 3
Loading model...


...

Using device: cuda

Evaluating: vit on atlas_isic
Model type: GLIDE-augmented training
Running on CUDA GPU: 3
Loading model...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading test dataset from: /data/team01/ds340w/datasets/Atlas_ISIC_split/test


Resolving data files:   0%|          | 0/962 [00:00<?, ?it/s]

Using device: cuda

ACCURACY COMPARISON: Original Training vs GLIDE-Augmented Training
dataset atlas_isic              ham         
variant      GLIDE Original   GLIDE Original
model                                       
dino        0.8773   0.8836  0.9051   0.9071
swin        0.8919   0.8940  0.8521   0.8721
vit         0.8191   0.8139  0.8931   0.8931

MACRO F1 SCORE COMPARISON: Original vs GLIDE
dataset atlas_isic              ham         
variant      GLIDE Original   GLIDE Original
model                                       
dino        0.8574   0.8617  0.8447   0.8361
swin        0.8713   0.8756  0.7946   0.7945
vit         0.7925   0.7988  0.8167   0.8069

WEIGHTED F1 SCORE COMPARISON: Original vs GLIDE
dataset atlas_isic              ham         
variant      GLIDE Original   GLIDE Original
model                                       
dino        0.8744   0.8807  0.9050   0.9073
swin        0.8894   0.8916  0.8627   0.8770
vit         0.8163   0.8118  0.8901   0.8903

RESULTS